> 在线讲义：[第 18 章 · 训练稳定性：注意力 logit 越涨越大、输出 logit 发散；QK-norm 和 z-loss 各治什么](https://wikieden.github.io/beyond-transformer/lessons/18-stability.html) · 本书《超越 Transformer》

# 第 18 章 · 训练稳定性：注意力 logit 越涨越大、输出 logit 发散；QK-norm 和 z-loss 各治什么

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/18-stability.html) · 仓库里 `beyond-transformer/notes/lessons/18-stability.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 logit 增长 | ∝ c² | `attn_stats(c)` | 2.84 → 181.76，熵 2.42 → 0.02 |
| 第二步 QK-norm | Norm(q)·Norm(k) | `attn_stats(c, qk_norm=True)` | 不随 c 变 |
| 第三步 z-loss | λ log²Z | `z_loss(logits)` | 0.090、0、0.160 |

QK-norm 先把 q、k 归一化，logit 有上界；输出 logit 发散（log Z 漂走）和 z-loss = λ·log²Z。
三个宽度 × 六个学习率的“LR 敏感度”实验、加不加 QK-norm / z-loss 的对比是里程碑 T-M4。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
D, N = 64, 16

**函数 `inputs`**：n 个已经过 LayerNorm 的 token（每个向量均值 0、方差 1）。

In [ ]:
def inputs(seed=0, n=N, d=D):
    """n 个已经过 LayerNorm 的 token（每个向量均值 0、方差 1）。"""
    x = np.random.default_rng(seed).normal(size=(n, d))
    return (x - x.mean(1, keepdims=True)) / x.std(1, keepdims=True)

**函数 `weights`**：小工具。

In [ ]:
def weights(seed=1, d=D):
    rng = np.random.default_rng(seed)
    return rng.normal(0, 1 / np.sqrt(d), (d, d)), rng.normal(0, 1 / np.sqrt(d), (d, d))

**函数 `rms`**：小工具。

In [ ]:
def rms(v):
    return v / np.sqrt((v * v).mean(-1, keepdims=True))

**函数 `attn_stats`**：W_Q、W_K 都乘 scale（训练中权重变大）。返回最大 |logit| 和每行 softmax 的平均熵（nats）、平均最大权重。

In [ ]:
def attn_stats(scale, qk_norm=False, seed=0):
    """W_Q、W_K 都乘 scale（训练中权重变大）。返回最大 |logit| 和每行 softmax 的平均熵（nats）、平均最大权重。"""
    x = inputs(seed)
    wq, wk = weights()
    q, k = x @ (scale * wq).T, x @ (scale * wk).T
    if qk_norm:
        q, k = rms(q), rms(k)
    s = q @ k.T / np.sqrt(D)
    p = np.exp(s - s.max(1, keepdims=True))
    p /= p.sum(1, keepdims=True)
    ent = float(-(p * np.log(p + 1e-30)).sum(1).mean())
    return float(np.abs(s).max()), ent, float(p.max(1).mean())

**函数 `z_loss`**：小工具。

In [ ]:
def z_loss(logits, lam=1e-4):
    z = np.log(np.exp(logits - logits.max()).sum()) + logits.max()
    return float(z), float(lam * z * z)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 注意力 logit 随权重平方增长")
for c in (1, 2, 4, 8):
    mx, ent, pm = attn_stats(c)
    print(f"  W_Q、W_K 乘 {c}：最大 |logit| {mx:7.2f}，平均熵 {ent:.3f}（均匀是 {np.log(N):.3f}），平均最大权重 {pm:.3f}")
print("\n第二步 QK-norm：先把 q、k 的均方根归一到 1")
for c in (1, 8):
    mx, ent, pm = attn_stats(c, qk_norm=True)
    print(f"  乘 {c}：最大 |logit| {mx:.2f}（上界 √d = {np.sqrt(D):.0f}），平均熵 {ent:.3f}")

In [ ]:
print("\n第三步 z-loss：把 log Z 拉回 0")
for lg in ([30.0, 25.0, 20.0], [0.0, -5.0, -10.0], [-40.0, -45.0, -50.0]):
    z, zl = z_loss(np.array(lg))
    print(f"  logits {lg}：log Z = {z:.4f}，z-loss（λ = 1e-4）= {zl:.6f}；softmax 都是 {np.round(np.exp(np.array(lg) - z), 4).tolist()}")

## 3. 代码里才看得到的

- **QK-norm 的位置**：这里在点积之前 norm；真实模型里还有 RoPE，一般是先 norm 再转（norm 改长度，旋转不改长度，顺序反过来结果也差不多）。
- **只模拟了“权重已经变大”的结果**：训练过程中 logit 怎么一步步长大、加了 QK-norm 后高学习率端不再发散，是里程碑 T-M4。

## 4. 自己改一改

- 把 `attn_stats` 的 D 改成 256，同样放大 4 倍，熵掉得更快还是更慢？
- 给 `rms` 乘一个可学缩放 g = 3（不再是 1），QK-norm 后的最大 logit 上界变成多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 两人嗓门各翻倍，吵架变四倍：W 大 c 倍，logit 大 c 方；softmax 塌成独热，熵坍缩。
> QK 先 norm 再点积，logit 有上界根号 d。
> softmax 不管平移，log Z 后期漂走；z-loss 万分之一乘 log Z 平方，拉回零、不改概率。
> 工业组合：RMSNorm、输出 norm、QK-norm、z-loss、大底数 RoPE；小模型大学习率当代理。